In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = '<YOUR_API_KEY>'
BASE_URL = "https://www.googleapis.com/youtube/v3"

In [3]:
channels_df= pd.read_csv('../channels.csv')
all_channel_ids = channels_df['channel_id']
len(all_channel_ids)

453

In [4]:
# Gather content details information using channel ids retrieved above
endpoint="channels"
channel_url=f"{BASE_URL}/{endpoint}"

content_details = []

# Fetch channel details in batches of 50, as the YouTube API accepts up to 50 channel IDs per request
for batch in range(0,len(all_channel_ids),50):
    params_channels = {
        "key": API_KEY,
        "part": "contentDetails",
        "id":  ','.join(list(all_channel_ids)[batch:batch+50])
    }
    response=requests.get(channel_url,params=params_channels).json()
    if "items" in response:
        content_details.extend(response["items"])
len(content_details)

453

In [5]:
channel_upload_playlist = [{
                             "channel_id":content['id'],
                             "uploads_playlist_id": content["contentDetails"]["relatedPlaylists"].get("uploads")
                           }
                           for content in content_details
                          ]
len(channel_upload_playlist)

453

In [6]:
endpoint="playlistItems"
upload_playlist_url = f"{BASE_URL}/{endpoint}"

upload_playlist_details=[]

for i in range(0,len(channel_upload_playlist)):
    params_upload_playlist = {
        "key": API_KEY,
        "part": "contentDetails,snippet",
        "playlistId": channel_upload_playlist[i]['uploads_playlist_id'],
        "maxResults":50
    }
    while True:
        response= requests.get(upload_playlist_url, params=params_upload_playlist)
        data= response.json()
        upload_playlist_details.extend(data.get('items',[]))
        if(data.get('nextPageToken')):
            params_upload_playlist['pageToken'] = data.get('nextPageToken')
        else:
            break

In [8]:
len(upload_playlist_details)
# upload_playlist_details[0]

31358

In [9]:
upload_playlist_df = pd.DataFrame({
                                    "channel_id":[detail['snippet']['channelId'] for detail in upload_playlist_details],
                                    "upload_playlist_id":[detail['snippet']['playlistId'] for detail in upload_playlist_details],
                                    "video_id":[detail['contentDetails']['videoId'] for detail in upload_playlist_details]
                                  })

In [10]:
upload_playlist_df.to_csv('upload_playlist.csv')

In [15]:
filtered_df = upload_playlist_df[upload_playlist_df["video_id"].str.startswith('-')]

In [16]:
filtered_df

,channel_id,upload_playlist_id,video_id
77,UCjz2kAizPuWCgNpMIe1LXzg,UUjz2kAizPuWCgNpMIe1LXzg,-yrNBxXVUQY
129,UCjz2kAizPuWCgNpMIe1LXzg,UUjz2kAizPuWCgNpMIe1LXzg,-JMaDtX23ao
178,UCuPwTACOC_FCSwg0qli98TA,UUuPwTACOC_FCSwg0qli98TA,---MpbQM4vg
186,UCuPwTACOC_FCSwg0qli98TA,UUuPwTACOC_FCSwg0qli98TA,-5vLw02aYd0
237,UCuPwTACOC_FCSwg0qli98TA,UUuPwTACOC_FCSwg0qli98TA,--7dklyymes
...,...,...,...
31028,UC9CYT9gSNLevX5ey2_6CK0Q,UU9CYT9gSNLevX5ey2_6CK0Q,-5TZt02kY0A
31063,UCckltLEhFLv8Xz_lQhYfwmg,UUckltLEhFLv8Xz_lQhYfwmg,-L-eGnMxiWc
31199,UCIkpWSONvFy829TYTcPJxAg,UUIkpWSONvFy829TYTcPJxAg,-DxbIWQ1jiM
31225,UCdZPINchtQ6JJn8rbkr12eg,UUdZPINchtQ6JJn8rbkr12eg,-Y6t1-5CQ_I
